# 00_core — Create canonical splits and TF-IDF vectorizer

Owner: Keana Gindlesperger

Reads: stanfordnlp/imdb (Hugging Face)
Writes: data/processed/splits.parquet, artifacts/tfidf_vectorizer.joblib

Notes: This notebook creates the canonical `splits.parquet` used by the pipeline
and fits the TF-IDF vectorizer on the `fit` split only. The notebook is written
to be explicit about seeds and artifact locations so downstream notebooks can
rely on the immutable disk-handoff contract.


In [ ]:
# Bootstrap import of src/shared.py (follows repo guidance)
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
from shared import SEED, PATHS, preprocess_text, fit_and_save_vectorizer, SPLIT_SIZES, PREDICTION_SCHEMA
import pandas as pd
import numpy as np
import os

print('SEED =', SEED)
print('splits target =', PATHS['splits_parquet'])
print('tfidf target =', PATHS['tfidf'])
print('split sizes =', SPLIT_SIZES)

print('prediction schema =', PREDICTION_SCHEMA)

SEED = 42
splits target = C:\Users\admin\Desktop\AAI501\movie-review-sentiment\data\processed\splits.parquet
tfidf target = C:\Users\admin\Desktop\AAI501\movie-review-sentiment\artifacts\tfidf_vectorizer.joblib


## Dependencies
The code below uses the Hugging Face `datasets` library to download `stanfordnlp/imdb`
and requires a parquet engine such as `pyarrow` to write parquet files. If these
are not installed in the environment, run `uv add datasets pyarrow` and then `uv sync`
before re-running this notebook.


In [4]:
# Attempt to import datasets and provide a friendly error if missing
try:
    from datasets import load_dataset
except Exception as e:
    raise RuntimeError(
        "Missing dependency: `datasets`. Install with `uv add datasets` then `uv sync`."
    ) from e

# Load the HF dataset (may take a while the first time)
print('Downloading stanfordnlp/imdb from Hugging Face...')
ds = load_dataset('stanfordnlp/imdb')
print('Dataset splits:', ds.keys())


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

c:\Users\admin\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\admin\.cache\huggingface\hub\datasets--stanfordnlp--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Dataset splits: dict_keys(['train', 'test', 'unsupervised'])


In [5]:
# Convert the HF dataset to a single pandas DataFrame (concatenate splits if needed)
def dataset_to_df(ds):
    # Accept either a single Split or a dict-like DatasetDict
    if isinstance(ds, dict):
        parts = []
        for k in ds.keys():
            parts.append(pd.DataFrame(ds[k]))
        df = pd.concat(parts, ignore_index=True)
    else:
        df = pd.DataFrame(ds)
    # Expect columns: 'text' and 'label' (0/1)
    df = df.rename(columns={c: c for c in df.columns})
    return df[['text', 'label']]

df_all = dataset_to_df(ds)
print('Total examples available:', len(df_all))


Total examples available: 100000


In [ ]:
# Create the canonical splits using the shared split-size contract
# Shuffle deterministically using SEED
N_fit = SPLIT_SIZES['fit']
N_val = SPLIT_SIZES['val']
N_test = SPLIT_SIZES['test']
N_total = N_fit + N_val + N_test
if len(df_all) < N_total:
    raise RuntimeError(f'Not enough examples ({len(df_all)}) to extract {N_total} total examples')

rng = np.random.RandomState(SEED)
perm = rng.permutation(len(df_all))
idx = perm[:N_total]
df_sel = df_all.iloc[idx].reset_index(drop=True).copy()

# Assign splits in order: first N_fit -> fit, next N_val -> val, next N_test -> test
splits = (['fit'] * N_fit) + (['val'] * N_val) + (['test'] * N_test)
df_sel['split'] = splits
# Create an id column (stable, reproducible)
df_sel['id'] = [f'ex-{i:06d}' for i in range(len(df_sel))]

# Reorder columns to the canonical contract
assert 'id' in PREDICTION_SCHEMA, 'Prediction schema must include id'
df_out = df_sel[['id', 'text', 'label', 'split']].copy()
print('Using prediction schema:', PREDICTION_SCHEMA)

# Ensure parent exists and write parquet
p = PATHS['splits_parquet']
p.parent.mkdir(parents=True, exist_ok=True)
try:
    df_out.to_parquet(p)
    print('Wrote splits.parquet ->', p)
except Exception as e:

    raise RuntimeError('Failed to write parquet. Ensure pyarrow is installed (uv add pyarrow)') from e

Wrote splits.parquet -> C:\Users\admin\Desktop\AAI501\movie-review-sentiment\data\processed\splits.parquet


In [7]:
# Fit TF-IDF on the 'fit' split and save the artifact intentionally
texts_fit = df_out.loc[df_out['split'] == 'fit', 'text'].astype(str).tolist()
print('Fitting TF-IDF on', len(texts_fit), 'documents (fit split)')
# This will write PATHS['tfidf']. Pass overwrite=True here for the initial run.
vec = fit_and_save_vectorizer(texts_fit, overwrite=True)
print('TF-IDF fitted and saved to', PATHS['tfidf'])


Fitting TF-IDF on 10000 documents (fit split)
TF-IDF fitted and saved to C:\Users\admin\Desktop\AAI501\movie-review-sentiment\artifacts\tfidf_vectorizer.joblib


## Summary
The canonical splits parquet and TF-IDF artifact have been created. Downstream notebooks
should call `from shared import PATHS, load_features` and use the saved artifacts.


In [8]:
print('Artifacts:')
print('  splits ->', PATHS['splits_parquet'])
print('  tfidf  ->', PATHS['tfidf'])


Artifacts:
  splits -> C:\Users\admin\Desktop\AAI501\movie-review-sentiment\data\processed\splits.parquet
  tfidf  -> C:\Users\admin\Desktop\AAI501\movie-review-sentiment\artifacts\tfidf_vectorizer.joblib
